# Week 06 · Sentiment Analysis & Text Classification
**ISB02306504 – Social Media Mining** · Information Systems for Business · Odd Semester 2026/2027

**Full** datasets:

| | Case 1 – Sentiment analysis | Case 2 – Text classification |
|---|---|---|
| Data | 617k Google Play reviews of Indonesian government apps | 31k social media comments |
| Target | Negative / Neutral / Positive (multiclass, imbalanced) | Judi online promotion vs not (binary) |
| Main lessons | label quality, duplicates & leakage, baselines, macro-F1, CV + GridSearch | obfuscated text, auditing someone else's cleaning, character n-grams, precision-first thresholds |

Both cases follow the same workflow from the slides:

`load → EDA → label check → preprocess → deduplicate → split → vectorize → baselines (NB / LR / SVM) → evaluate → cross-validate & tune → final test → error analysis`

| Section | Slides |
|---|---|
| 0. Setup and data loading | – |
| Case 1 · EDA, labeling, preprocessing | 3–7 |
| Case 1 · Split, leakage, pipeline | 8–10 |
| Case 1 · Naive Bayes, Logistic Regression, SVM | 12–20 |
| Case 1 · Evaluation (confusion matrix, macro/micro/weighted) | 22–29 |
| Case 1 · Cross-validation, GridSearchCV, model selection | 31–37 |
| Case 2 · Judol detection (all of the above, binary, precision vs recall) | 3–37 |
| Discussion exercise and homework | 39–41 |

**Runtime:** standard CPU runtime. *Run all* takes roughly 10–15 minutes on Colab; the longest cells print how long they took.

---
## 0. Setup
Everything used here is preinstalled on Colab (pandas, scikit-learn, matplotlib, nltk).

In [ ]:
import re, time, json, unicodedata, urllib.request, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

import nltk
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve, make_scorer)
import sklearn

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 160)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "font.size": 9})
SENT_COLORS = {"Negative": "#eb6834", "Neutral": "#8a8984", "Positive": "#2a78d6"}
JUDOL_COLORS = {0: "#2a78d6", 1: "#eb6834"}

print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

### Loading big files responsibly
The review file is about **100 MB**. Three habits keep this manageable:

1. **Download once** to local disk and reuse the copy, instead of reading the URL on every run.
2. **Peek before loading**: read a few rows to learn the columns and types.
3. **Load only what you need**: select columns (`usecols`), use compact types, and stream the file in chunks so you can see progress and control memory. We still end up with **every row**.

In [ ]:
BASE_URL = ("https://raw.githubusercontent.com/satriaadi9/smm/refs/heads/main/"
            "w6%20-%20sentiment%20analysis%20%26%20text%20classification/")
GOV_URL = BASE_URL + "indonesian-government-apps-review.csv"
JUDOL_URL = BASE_URL + "judol-dataframe.csv"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def download_once(url, dest):
    """Download url to dest unless a complete copy already exists."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"{dest.name}: using cached copy ({dest.stat().st_size / 1e6:.1f} MB)")
        return dest
    tmp = dest.with_name(dest.name + ".part")
    shown = set()
    def progress(blocks, block_size, total):
        if total > 0:
            pct = min(100, blocks * block_size * 100 // total) // 20 * 20
            if pct not in shown:
                shown.add(pct); print(f"  {dest.name}: {pct}%")
    t0 = time.perf_counter()
    urllib.request.urlretrieve(url, tmp, reporthook=progress)
    tmp.rename(dest)
    print(f"{dest.name}: downloaded {dest.stat().st_size / 1e6:.1f} MB in {time.perf_counter() - t0:.0f}s")
    return dest

GOV_PATH = download_once(GOV_URL, DATA_DIR / "indonesian-government-apps-review.csv")
JUDOL_PATH = download_once(JUDOL_URL, DATA_DIR / "judol-dataframe.csv")

---
# Case 1 · Sentiment of Indonesian government app reviews

**Business question:** the ministries behind apps such as *SATUSEHAT*, *JMO* (BPJS Ketenagakerjaan), *Mobile JKN* and *MyPertamina* want to monitor user satisfaction automatically. Each review must be classified as **Negative, Neutral or Positive**, which makes this a document-level, multiclass sentiment task (slides 4–5).

## 1.1 Peek, then load

In [ ]:
peek = pd.read_csv(GOV_PATH, nrows=5)
print("Columns:", list(peek.columns))
display(peek)

| Column | Meaning | Keep? |
|---|---|---|
| `Unnamed: 0` | old index | no |
| `app` | which government app | yes, for per-app analysis |
| `content` | review text (Indonesian) | **yes, the input `x`** |
| `translation` | machine translation to English | no: it is long text we do not need, and we model the original language |
| `score` | star rating 1–5 | yes, for checking the label |
| `at`, `appVersion` | timestamp, app version | yes, for EDA |
| `labelScoreBase` | sentiment label | **yes, the target `y`** |

How much do we save by skipping columns? Measure it on the first 50k rows:

In [ ]:
USECOLS = ["app", "content", "score", "at", "appVersion", "labelScoreBase"]
all_cols = pd.read_csv(GOV_PATH, nrows=50_000)
lean = pd.read_csv(GOV_PATH, nrows=50_000, usecols=USECOLS, dtype={"score": "int8"})
mb = lambda d: d.memory_usage(deep=True).sum() / 1e6
print(f"50k rows, all columns : {mb(all_cols):6.1f} MB")
print(f"50k rows, needed only : {mb(lean):6.1f} MB")
del all_cols, lean

In [ ]:
t0 = time.perf_counter()
chunks, n_rows = [], 0
for chunk in pd.read_csv(GOV_PATH, usecols=USECOLS, dtype={"score": "int8"}, chunksize=100_000):
    chunks.append(chunk)
    n_rows += len(chunk)
    print(f"  loaded {n_rows:,} rows", end="\r")
gov = pd.concat(chunks, ignore_index=True)
del chunks

gov = gov.rename(columns={"labelScoreBase": "label"})
gov["app"] = gov["app"].astype("category")
gov["label"] = pd.Categorical(gov["label"], categories=["Negative", "Neutral", "Positive"])
gov["at"] = pd.to_datetime(gov["at"])
print(f"\nLoaded {len(gov):,} reviews in {time.perf_counter() - t0:.1f}s, {mb(gov):.0f} MB in memory")
gov.info()

## 1.2 Exploratory data analysis
Before any modelling: how big is each class, where does the data come from, and what do the texts look like?

In [ ]:
print("Missing values:\n", gov.isna().sum().to_string(), "\n")
print("Date range:", gov["at"].min().date(), "→", gov["at"].max().date())

label_counts = gov["label"].value_counts().reindex(SENT_COLORS)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1, 1.5]})
axes[0].bar(label_counts.index, label_counts.values, color=[SENT_COLORS[k] for k in label_counts.index], width=0.6)
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v, f"{v:,}\n({v / len(gov):.1%})", ha="center", va="bottom", fontsize=8)
axes[0].set_title("Reviews per label"); axes[0].set_ylim(0, label_counts.max() * 1.25)

by_app = pd.crosstab(gov["app"], gov["label"], normalize="index").reindex(columns=SENT_COLORS)
by_app = by_app.loc[gov["app"].value_counts().index[::-1]]
left = np.zeros(len(by_app))
for lab in SENT_COLORS:
    axes[1].barh(by_app.index.astype(str), by_app[lab], left=left, color=SENT_COLORS[lab], label=lab,
                 edgecolor="white", linewidth=1)
    left += by_app[lab].values
n_per_app = gov["app"].value_counts()
axes[1].set_yticks(range(len(by_app)), [f"{a} (n={n_per_app[a]:,})" for a in by_app.index])
axes[1].set_xlim(0, 1); axes[1].set_title("Label share per app"); axes[1].legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.32))
plt.tight_layout(); plt.show()

- **Neutral is a small minority** (about 6%). Accuracy will be misleading; we will use **macro-F1** (slides 23 and 28).
- Apps differ a lot: SATUSEHAT and JMO lean negative, KAI and BMKG are tiny. Per-app results will matter later.

### Where do the labels come from?

In [ ]:
display(pd.crosstab(gov["score"], gov["label"]))

The label is a **rule on the star rating**: 1–2 stars = Negative, 3 = Neutral, 4–5 = Positive. Nobody read the texts. This is *weak supervision*: cheap and large-scale, but the model will learn "how many stars would this text get", which is not always the same as the sentiment of the text (slide 6: *the classifier learns the decisions encoded in the annotations*).

In [ ]:
words = gov["content"].fillna("").str.split().str.len()
gov["n_words"] = words.astype("int16")
print(gov.groupby("label", observed=True)["n_words"].describe()[["mean", "25%", "50%", "75%", "max"]].round(1))

fig, ax = plt.subplots(figsize=(8, 3))
bins = np.arange(0, 61, 2)
for lab, c in SENT_COLORS.items():
    ax.hist(gov.loc[gov["label"] == lab, "n_words"].clip(upper=60), bins=bins, histtype="step", lw=2,
            color=c, label=lab, density=True)
ax.set_xlabel("words per review (clipped at 60)"); ax.set_ylabel("share of reviews"); ax.legend()
ax.set_title("Review length by label")
plt.tight_layout(); plt.show()

Half of all reviews have **4 words or fewer**, and positive reviews are the shortest ("Mantap", "Sangat membantu"). Negative reviews are longer because people explain what went wrong.

### Duplicates

In [ ]:
dup_share = gov["content"].duplicated().mean()
print(f"Exact duplicate reviews: {gov['content'].duplicated().sum():,} ({dup_share:.1%})")
top_dups = gov["content"].value_counts().head(10).rename("copies").to_frame()
top_dups["label of first copy"] = gov.drop_duplicates("content").set_index("content").loc[top_dups.index, "label"].values
display(top_dups)

conflicts = gov.groupby("content", observed=True)["label"].nunique()
print(f"Identical texts that carry more than one label: {(conflicts > 1).sum():,}")
display(gov[gov["content"] == "Ok"]["label"].value_counts().to_frame("'Ok' reviews by label"))

More than a third of the rows are copies of short, generic reviews, and the same text often appears with different labels ("Ok" can come with 1 star or 5 stars). This has two consequences we deal with in 1.5:
- duplicates can sit in both train and test, which is **leakage** (slide 9);
- identical inputs with different labels are **label noise** that no model can resolve.

### Volume over time

In [ ]:
per_year = pd.crosstab(gov["at"].dt.year, gov["label"]).reindex(columns=SENT_COLORS)
fig, ax = plt.subplots(figsize=(8, 3))
for lab, c in SENT_COLORS.items():
    ax.plot(per_year.index, per_year[lab], marker="o", color=c, label=lab, lw=2)
ax.set_title("Reviews per year"); ax.set_ylabel("reviews"); ax.legend()
plt.tight_layout(); plt.show()

Almost all reviews are from 2020 onward (COVID-era apps and BPJS). A model trained on this period may drift when the apps, and the complaints, change. Keep that in mind when you deploy.

### Which words separate the classes?
Raw frequency lists are dominated by words common to every class (`aplikasi`, `yang`). A better view is the **log-odds ratio**: how much more likely a word is in one class than in the others (with +1 smoothing so rare words do not explode).

In [ ]:
SW_ID = set(stopwords.words("indonesian")) - {"tidak", "belum", "bukan", "kurang", "sangat", "baik", "bisa"}
tok = lambda s: [w for w in re.findall(r"[a-z]+", s.lower()) if w not in SW_ID and len(w) > 2]

sample = gov.sample(80_000, random_state=SEED)
class_counts = {lab: Counter(w for t in sample.loc[sample["label"] == lab, "content"].dropna() for w in tok(t))
                for lab in SENT_COLORS}
total = sum(class_counts.values(), Counter())
V = len(total)

def log_odds(lab, min_count=100):
    c, rest = class_counts[lab], total - class_counts[lab]
    n_c, n_r = sum(c.values()), sum(rest.values())
    return pd.Series({w: np.log((c[w] + 1) / (n_c + V)) - np.log((rest[w] + 1) / (n_r + V))
                      for w in c if total[w] >= min_count}).nlargest(12)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, lab in zip(axes, SENT_COLORS):
    s = log_odds(lab)[::-1]
    ax.barh(s.index, s.values, color=SENT_COLORS[lab]); ax.set_title(f"Most {lab}-specific words")
    ax.set_xlabel("log-odds vs other classes")
plt.tight_layout(); plt.show()

- **Negative** words are insults and strong complaints (*sampah, menyusahkan, cacat, buruk, ribet*).
- **Positive** words are praise, in many spellings: *mantap, mantaap, mantab, mantaf, mntap, mantul*. This is why normalising elongation and slang matters.
- **Neutral** has the weakest scores (look at the x-axis) and mixed words (*lumayan, kadang, tambahin*): suggestions and "it's okay" reviews. Expect Neutral to be the hardest class.

## 1.3 Label quality check
Star-derived labels can contradict the text. Let's count obvious cases.

In [ ]:
complaint = r"\b(?:error|gagal|lemot|tidak bisa|gak bisa|ga bisa|susah|kecewa)\b"
praise = r"\b(?:mantap|bagus|membantu|terima kasih|keren|mudah)\b"
text_lower = gov["content"].fillna("").str.lower()

negated_praise = r"\b(?:tidak|gak|ga|nggak|kurang)\s+(?:mantap|bagus|membantu|keren|mudah)"
five = text_lower[gov["score"] == 5]
one = text_lower[gov["score"] == 1]
five_star_complaints = gov.loc[five[five.str.contains(complaint)].index]
one_hits = one[one.str.contains(praise) & ~one.str.contains(complaint) & ~one.str.contains(negated_praise)]
one_star_praise = gov.loc[one_hits.index]
print(f"5-star reviews with complaint words: {len(five_star_complaints):,}")
print(f"1-star reviews with (non-negated) praise words and no complaint words: {len(one_star_praise):,}")
display(five_star_complaints.sample(5, random_state=1)[["app", "score", "label", "content"]])
display(one_star_praise.sample(5, random_state=1)[["app", "score", "label", "content"]])

Keyword counts are rough (some 1-star "praise" is sarcasm such as *"membantu… membantu bikin pusing"*), but the examples are clear enough. Some are genuinely mixed ("bagus tapi sering error"), some are users clicking the wrong star, and some are sarcastic. If we were labelling by hand, a **labeling guide** (slide 6) would settle them:

| Label | Operational definition | Yes | No |
|---|---|---|---|
| Negative | dissatisfaction, complaint, failure report | "otp tidak masuk masuk" | "kapan update?" (question) |
| Neutral | no clear evaluation, questions, suggestions, mixed with no dominant side | "tolong tambah fitur dark mode" | "bagus tapi lemot" (decide: complaint dominates → Negative) |
| Positive | satisfaction, praise, thanks | "sangat membantu" | "makasih ya, akhirnya bisa juga setelah 3 hari" (sarcasm policy needed) |

**Decision for this lab:** we keep the star-based labels, because relabelling 600k reviews is not possible in class, but we treat the result as *"predicting the star group from the text"* and we will see the noise again in the error analysis. When annotators (or stars) disagree: inspect, revisit the definition, add an edge-case rule, relabel, record the decision (slide 7).

## 1.4 Preprocessing
Week 5 showed that preprocessing is a modelling decision. For this task:

| Step | Choice | Reason |
|---|---|---|
| Unicode NFKC, lowercase | yes | merge `Bagus`/`bagus`, fancy fonts |
| URLs | remove | rare in app reviews, no sentiment |
| Elongation `bagusss` | reduce | merge spellings |
| Slang `gk, yg, bgt` | normalize with a small dictionary | merge spellings; negation goes to `tidak` |
| Punctuation | remove (keep `!`/`?` as tokens) | TF-IDF works on words |
| Stopwords | **do not remove** | `tidak`, `belum`, `kurang` carry sentiment; TF-IDF already down-weights common words |
| Stemming | no | slow on 380k texts, and `membantu`/`bantuan` behave differently; try it as homework |

We apply the function to each **unique** text once and map the result back, which is about 40% less work here.

In [ ]:
SLANG = {"gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak", "enggak": "tidak",
         "tdk": "tidak", "kagak": "tidak", "yg": "yang", "dgn": "dengan", "utk": "untuk", "sy": "saya",
         "sdh": "sudah", "udh": "sudah", "udah": "sudah", "blm": "belum", "bgt": "banget", "aja": "saja",
         "tp": "tapi", "krn": "karena", "jd": "jadi", "bs": "bisa", "lg": "lagi", "dr": "dari", "apk": "aplikasi",
         "aplikasih": "aplikasi", "trs": "terus", "klo": "kalau", "kalo": "kalau", "mksh": "terima kasih",
         "makasih": "terima kasih", "thx": "terima kasih", "tq": "terima kasih"}
URL_RE = re.compile(r"https?://\S+|www\.\S+")
ELONG_RE = re.compile(r"([a-z])\1{2,}")
TOKEN_RE = re.compile(r"[a-z0-9]+|[!?]")

def clean_review(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = URL_RE.sub(" ", text)
    text = ELONG_RE.sub(r"\1", text)
    tokens = []
    for t in TOKEN_RE.findall(text):
        tokens.extend(SLANG.get(t, t).split())
    return " ".join(tokens)

for t in ["Lelet....", "Aplikasinya GAK BISA dibuka!!! sdh 3x install ulang 😡", "Mantaaap, sangat membantu bgt 👍"]:
    print(f"{t!r:60} → {clean_review(t)!r}")

In [ ]:
t0 = time.perf_counter()
unique_texts = gov["content"].dropna().unique()
clean_map = {t: clean_review(t) for t in unique_texts}
gov["clean"] = gov["content"].map(clean_map).fillna("")
print(f"Cleaned {len(unique_texts):,} unique texts (for {len(gov):,} rows) in {time.perf_counter() - t0:.1f}s")

## 1.5 Deduplicate and resolve conflicting labels
One row per unique cleaned text. If copies disagree, keep the **majority label**; if there is no majority (for example 1 Positive vs 1 Negative), drop the text. We keep `n_copies` so the information about popularity is not lost.

In [ ]:
d = gov.loc[gov["clean"].str.len() > 0, ["clean", "label"]]
counts = d.groupby(["clean", "label"], observed=True).size().unstack(fill_value=0)
total_copies = counts.sum(axis=1)
majority_share = counts.max(axis=1) / total_copies
keep = majority_share > 0.5

data = pd.DataFrame({"text": counts.index[keep],
                     "label": counts.idxmax(axis=1)[keep].values,
                     "n_copies": total_copies[keep].values})
data["label"] = pd.Categorical(data["label"], categories=list(SENT_COLORS))
first_app = gov.drop_duplicates("clean").set_index("clean")["app"]
data["app"] = first_app.reindex(data["text"]).values
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)      # shuffle once

print(f"Rows with text      : {len(d):,}")
print(f"Unique cleaned texts: {len(counts):,}")
print(f"Dropped (no majority label): {(~keep).sum():,}")
print(f"Modelling dataset   : {len(data):,}")

compare = pd.DataFrame({"all rows": d["label"].value_counts(normalize=True),
                        "after dedup": data["label"].value_counts(normalize=True)}).reindex(list(SENT_COLORS))
display((compare * 100).round(1).astype(str) + "%")

Deduplication **changes the class balance**: Positive drops from about 54% of rows to about a third, because the duplicates were mostly short positive reviews. The modelling dataset now has Negative as the majority class. This is the population the classifier will be evaluated on, so it is worth knowing.

## 1.6 Train / validation / test split
Each split answers a different question (slide 8):

- **Test (20%)** is locked away now and used **once** at the very end.
- The remaining **development set** is split again into **train** and **validation** for comparing baselines. Later, cross-validation runs inside the development set.

All splits are **stratified**, so class proportions stay the same (slide 9).

In [ ]:
dev, test = train_test_split(data, test_size=0.20, stratify=data["label"], random_state=SEED)
train, val = train_test_split(dev, test_size=0.20, stratify=dev["label"], random_state=SEED)

split_table = pd.DataFrame({name: part["label"].value_counts(normalize=True)
                            for name, part in [("train", train), ("validation", val), ("test", test)]})
split_table.loc["n rows"] = [len(train), len(val), len(test)]
display(split_table.reindex(list(SENT_COLORS) + ["n rows"]).style.format(
    lambda v: f"{v:,.0f}" if v > 1 else f"{v:.1%}"))

## 1.7 Baseline models
Rule of thumb from slide 20: **same split, same representation**. We fit one TF-IDF vectorizer on the training set, then train four classifiers on the same matrix:

- **Dummy (majority class):** the floor every model must beat.
- **Multinomial Naive Bayes:** fast probabilistic baseline.
- **Logistic Regression:** strong default for TF-IDF.
- **Linear SVM:** often strong on sparse text.

For a single train/validation split, fitting the vectorizer once on `train` and reusing it is equivalent to a `Pipeline`. For cross-validation we will switch to a real `Pipeline`.

In [ ]:
t0 = time.perf_counter()
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
X_train = tfidf.fit_transform(train["text"])
X_val = tfidf.transform(val["text"])
y_train, y_val = train["label"].astype(str), val["label"].astype(str)
print(f"TF-IDF: {X_train.shape[0]:,} docs × {X_train.shape[1]:,} features ({time.perf_counter() - t0:.0f}s)")

baselines = {
    "Dummy (majority)": DummyClassifier(strategy="most_frequent"),
    "Multinomial NB": MultinomialNB(alpha=1.0),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(C=1.0),
}
rows, val_preds, fitted = [], {}, {}
for name, clf in baselines.items():
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    pred = clf.predict(X_val)
    val_preds[name], fitted[name] = pred, clf
    rows.append({"model": name, "accuracy": accuracy_score(y_val, pred),
                 "macro P": precision_score(y_val, pred, average="macro", zero_division=0),
                 "macro R": recall_score(y_val, pred, average="macro"),
                 "macro F1": f1_score(y_val, pred, average="macro"),
                 "Neutral recall": recall_score(y_val, pred, labels=["Neutral"], average="macro"),
                 "train s": fit_s})
baseline_table = pd.DataFrame(rows).set_index("model")
display(baseline_table.style.format("{:.3f}").format("{:.1f}", subset=["train s"]))

Read the table from the top:
- The **dummy** model already reaches a decent accuracy by always answering "Negative", while macro-F1 shows it is useless. This is the *accuracy trap* from slide 23.
- All three real models beat the dummy by a wide margin. **Logistic Regression and Linear SVM** are close; **Naive Bayes** is fastest but weaker.
- **Neutral recall is very low for every model.** The classes are imbalanced and Neutral texts are genuinely ambiguous (see 1.3). We will try `class_weight="balanced"` during tuning.

## 1.8 Inside the three models
### Naive Bayes: prior × likelihood (slides 12–15)
Let's reproduce the slide-14 calculation for one review with a count-based Multinomial NB. Words never seen in training are simply ignored.

In [ ]:
count_vec = CountVectorizer(min_df=3)
nb_counts = MultinomialNB(alpha=1.0).fit(count_vec.fit_transform(train["text"]), y_train)
vocab, classes = count_vec.vocabulary_, list(nb_counts.classes_)

review = clean_review("aplikasinya lemot dan sering error, otp tidak masuk qwertyz")
words_in = [w for w in review.split() if w in vocab]
print("Review :", review)
print("Ignored (unseen):", [w for w in review.split() if w not in vocab])

contrib = pd.DataFrame({c: [nb_counts.feature_log_prob_[i, vocab[w]] for w in words_in]
                        for i, c in enumerate(classes)}, index=words_in)
contrib.loc["log prior P(c)"] = nb_counts.class_log_prior_
scores = contrib.sum()
posterior = np.exp(scores - scores.max()); posterior /= posterior.sum()
display(contrib.round(2))
print("log P(c) + Σ log P(w|c):", {str(k): round(float(v), 2) for k, v in scores.items()})
print("posterior:", {str(k): round(float(v), 3) for k, v in posterior.items()}, "→ prediction:", scores.idxmax())

Each row is a word's evidence. *lemot*, *error* and *otp* are much more likely under Negative, so their log-likelihoods are less negative there. Adding them up (logs turn the product into a sum) and normalising gives the posterior. The "naive" part is assuming the words are independent given the class.

### Logistic Regression: one weight per feature (slides 16–17)
For three classes, LR learns one weight vector per class. Positive weights push towards that class.

In [ ]:
lr = fitted["Logistic Regression"]
names = tfidf.get_feature_names_out()
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (i, cls) in zip(axes, enumerate(lr.classes_)):
    top = np.argsort(lr.coef_[i])[-12:]
    ax.barh(names[top], lr.coef_[i][top], color=SENT_COLORS[cls])
    ax.set_title(f"Strongest weights → {cls}"); ax.set_xlabel("coefficient")
plt.tight_layout(); plt.show()

example = clean_review("Tolong diperbaiki, sudah update tapi tetap tidak bisa login")
print(example, "→", {str(c): round(float(p), 3) for c, p in zip(lr.classes_, lr.predict_proba(tfidf.transform([example]))[0])})

This is why LR is popular in business settings: you can show a stakeholder *which words* push a review toward Negative. Three things to notice:
- Bigrams such as `tidak membantu` and `tidak ribet` get their own weights. Negation survives because we did not remove stopwords.
- The top Neutral features are `tiga`, `bintang tiga`, `3dulu`: users write *"bintang 3 dulu"* ("3 stars for now"). The text **mentions the label**. It is genuine signal here, but in other projects a feature that restates the target is a leakage warning sign.
- Some weights are about the topic, not sentiment (`rakyat`, `aplikasi` → Negative). They reflect what angry users talk about.

### Linear SVM: the C trade-off (slides 18–19)
Small `C` means stronger regularisation and a wider margin; large `C` fits the training data more strictly and can overfit. We sweep `C` on a 40k subsample of `train` and score on the full validation set.

In [ ]:
sub_idx = train.groupby("label", observed=True).sample(frac=40_000 / len(train), random_state=SEED).index
X_sub, y_sub = tfidf.transform(train.loc[sub_idx, "text"]), train.loc[sub_idx, "label"].astype(str)
Cs = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]
curve = []
for C in Cs:
    svm = LinearSVC(C=C, max_iter=5000).fit(X_sub, y_sub)
    curve.append((f1_score(y_sub, svm.predict(X_sub), average="macro"),
                  f1_score(y_val, svm.predict(X_val), average="macro")))
curve = np.array(curve)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(Cs, curve[:, 0], marker="o", color="#8a8984", label="training subset")
ax.plot(Cs, curve[:, 1], marker="o", color="#2a78d6", label="validation")
ax.set_xscale("log"); ax.set_xlabel("C (log scale)"); ax.set_ylabel("macro-F1"); ax.legend()
ax.set_title("Linear SVM: regularisation vs overfitting")
plt.tight_layout(); plt.show()
print("Best C on validation:", Cs[int(curve[:, 1].argmax())])

As `C` grows, the training score keeps rising while the validation score peaks and then falls: the model starts memorising. Choosing `C` is exactly what cross-validation and grid search are for (section 1.10).

## 1.9 Evaluation in detail
We look at the best baseline on the **validation** set (the test set is still locked).

In [ ]:
best_name = baseline_table["macro F1"].idxmax()
pred = val_preds[best_name]
print(f"Model: {best_name}\n")
print(classification_report(y_val, pred, digits=3))

labels = list(SENT_COLORS)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ConfusionMatrixDisplay.from_predictions(y_val, pred, labels=labels, ax=axes[0], cmap="Blues", colorbar=False,
                                        values_format=",")
axes[0].set_title("Counts")
ConfusionMatrixDisplay.from_predictions(y_val, pred, labels=labels, normalize="true", ax=axes[1], cmap="Blues",
                                        colorbar=False, values_format=".2f")
axes[1].set_title("Row-normalised (= recall per class)")
plt.tight_layout(); plt.show()

In a confusion matrix, rows are the **actual** class and columns the **predicted** class. Most Neutral reviews are predicted as Negative or Positive. That is understandable: a 3-star review is often "good but…".

### Macro vs micro vs weighted (slide 28)

In [ ]:
avg_table = pd.DataFrame({avg: {"precision": precision_score(y_val, pred, average=avg),
                                "recall": recall_score(y_val, pred, average=avg),
                                "F1": f1_score(y_val, pred, average=avg)}
                          for avg in ["macro", "micro", "weighted"]}).T
display(avg_table.round(3))
print("Note: micro-F1 equals accuracy for single-label multiclass:", round(accuracy_score(y_val, pred), 3))

- **Micro** pools every decision, so it is dominated by the big classes (and equals accuracy here).
- **Weighted** averages per-class scores by support: it also hides the weak Neutral class.
- **Macro** gives Neutral the same vote as the other classes, which is why it is the most honest single number for this task.

Performance per app, because a ministry only cares about its own app:

In [ ]:
val_frame = pd.DataFrame({"app": val["app"].astype(str).values, "y": y_val.values, "pred": pred})
per_app = pd.DataFrame([{"app": app, "n": len(g), "accuracy": (g["y"] == g["pred"]).mean(),
                         "macro F1": f1_score(g["y"], g["pred"], average="macro")}
                        for app, g in val_frame.groupby("app")]).set_index("app").sort_values("n", ascending=False)
display(per_app.style.format({"n": "{:,.0f}", "accuracy": "{:.3f}", "macro F1": "{:.3f}"}))

## 1.10 Cross-validation and hyperparameter tuning
### One split is not enough (slide 31)
Compare LR and SVM on three different random splits of the same 60k reviews:

In [ ]:
dev60 = dev.groupby("label", observed=True).sample(frac=60_000 / len(dev), random_state=SEED)
rows = []
for seed in [1, 2, 3]:
    tr, va = train_test_split(dev60, test_size=0.2, stratify=dev60["label"], random_state=seed)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(tr["text"])
    A, B = vec.transform(tr["text"]), vec.transform(va["text"])
    row = {"split": f"seed {seed}"}
    for name, clf in [("LR", LogisticRegression(max_iter=1000)), ("SVM", LinearSVC())]:
        row[name] = f1_score(va["label"].astype(str), clf.fit(A, tr["label"].astype(str)).predict(B), average="macro")
    rows.append(row)
display(pd.DataFrame(rows).set_index("split").round(3))

The scores move by about half a point from split to split. Here SVM wins every time, but with smaller datasets or closer models the ranking can flip (slide 31). **K-fold cross-validation** trains *k* times so that every example is used for validation exactly once, then averages (slides 32–33). **Stratified** k-fold keeps the class mix the same in every fold:

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_mix = pd.DataFrame({f"fold {k + 1}": dev60["label"].iloc[va_idx].value_counts(normalize=True)
                         for k, (_, va_idx) in enumerate(skf.split(dev60["text"], dev60["label"]))})
display((fold_mix.reindex(list(SENT_COLORS)) * 100).round(1))

### GridSearchCV with a Pipeline (slides 34–36)
The vectorizer lives **inside** the pipeline, so it is refitted on the training folds of every CV split: no leakage. We tune the slide's grid (n-grams × C) plus `class_weight`, scored by **macro-F1**.

To keep class time reasonable, the search runs on the 60k stratified subsample of the development set (12 settings × 5 folds = 60 fits). The winning settings are then refitted on the **full** development set.

In [ ]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=3, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000)),
])
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.1, 1, 10],
    "clf__class_weight": [None, "balanced"],
}
search = GridSearchCV(pipe, param_grid, scoring="f1_macro", cv=skf, n_jobs=-1, return_train_score=False)

t0 = time.perf_counter()
search.fit(dev60["text"], dev60["label"].astype(str))
print(f"Grid search finished in {time.perf_counter() - t0:.0f}s")

cv_results = (pd.DataFrame(search.cv_results_)
              [["param_tfidf__ngram_range", "param_clf__C", "param_clf__class_weight",
                "mean_test_score", "std_test_score", "rank_test_score"]]
              .sort_values("rank_test_score"))
display(cv_results.head(8).style.format({"mean_test_score": "{:.3f}", "std_test_score": "{:.3f}"}))
print("Best settings:", search.best_params_)

Look at both columns: the mean says how good a setting is, the standard deviation says how **stable** it is across folds. `class_weight="balanced"` usually wins on macro-F1 because it raises Neutral recall, at the cost of some Positive/Negative precision.

### Final model: refit on all development data, test once

In [ ]:
final_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=3, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000)),
]).set_params(**search.best_params_)

t0 = time.perf_counter()
final_pipe.fit(dev["text"], dev["label"].astype(str))
train_seconds = time.perf_counter() - t0
y_test = test["label"].astype(str)
test_pred = final_pipe.predict(test["text"])
print(f"Trained on {len(dev):,} reviews in {train_seconds:.0f}s\n")
print(classification_report(y_test, test_pred, digits=3))

untuned_pred = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)),
                         ("clf", LogisticRegression(max_iter=1000))]).fit(dev["text"], dev["label"].astype(str)).predict(test["text"])
print(f"Untuned LR macro-F1 on test: {f1_score(y_test, untuned_pred, average='macro'):.3f} | "
      f"tuned: {f1_score(y_test, test_pred, average='macro'):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 3.8))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, labels=list(SENT_COLORS), normalize="true", ax=ax,
                                        cmap="Blues", colorbar=False, values_format=".2f")
ax.set_title("Final model on the test set (row-normalised)")
plt.tight_layout(); plt.show()

## 1.11 Error analysis
Numbers tell you *how much* the model fails; examples tell you *why*. The most confident mistakes are especially informative, because they often point at label noise.

In [ ]:
proba = final_pipe.predict_proba(test["text"])
err = test.assign(pred=test_pred, confidence=proba.max(axis=1))
err = err[err["pred"] != err["label"].astype(str)]
print(f"Test errors: {len(err):,} of {len(test):,}")
display(err.nlargest(10, "confidence")[["app", "label", "pred", "confidence", "text"]])
display(err[err["label"] == "Neutral"].sample(6, random_state=SEED)[["label", "pred", "confidence", "text"]])

Many "confident errors" are reviews whose star rating contradicts the text, which is exactly the label noise found in 1.3. Better models will not fix that; better labels will.

## 1.12 Model selection and saving (slide 37)
Choose the simplest model that meets the need. The table below collects the evidence for that decision.

In [ ]:
best_cv = cv_results.iloc[0]
joblib.dump(final_pipe, "sentiment_gov_reviews_lr.joblib")
size_mb = Path("sentiment_gov_reviews_lr.joblib").stat().st_size / 1e6

t0 = time.perf_counter(); final_pipe.predict(test["text"].iloc[:10_000]); ms_per_1k = (time.perf_counter() - t0) / 10 * 1000

selection = pd.DataFrame({
    "criterion": ["Predictive quality", "Error profile", "Stability", "Interpretability", "Operational cost"],
    "question": ["Meets the target metric?", "Are costly mistakes controlled?", "Is CV performance consistent?",
                 "Can stakeholders see why?", "Latency, memory, retraining?"],
    "evidence": [f"test macro-F1 = {f1_score(y_test, test_pred, average='macro'):.3f}",
                 f"Neutral recall = {recall_score(y_test, test_pred, labels=['Neutral'], average='macro'):.2f}; "
                 f"Negative recall = {recall_score(y_test, test_pred, labels=['Negative'], average='macro'):.2f}",
                 f"CV std = {best_cv['std_test_score']:.3f}",
                 "per-word weights (section 1.8)",
                 f"train {train_seconds:.0f}s, {ms_per_1k:.0f} ms per 1,000 reviews, file {size_mb:.0f} MB"],
})
display(selection.set_index("criterion"))

In [ ]:
model = joblib.load("sentiment_gov_reviews_lr.joblib")
new_reviews = ["Aplikasi sangat membantu, proses klaim jadi cepat",
               "Sudah 3 hari OTP tidak masuk, tolong diperbaiki",
               "Kapan fitur cetak kartu ditambahkan?",
               "Lumayan, tapi kadang masih lemot"]
cleaned = [clean_review(r) for r in new_reviews]
out = pd.DataFrame(model.predict_proba(cleaned), columns=model.classes_, index=new_reviews).round(2)
out["prediction"] = model.predict(cleaned)
display(out)

The question about a new feature ("Kapan fitur…") should be Neutral but is predicted Positive: Neutral remains the weak spot.

Remember to clean new text with the **same** `clean_review` function used in training. In production you would put the cleaning step inside the pipeline (for example with `FunctionTransformer`) so it can never be forgotten.

---
# Case 2 · Detecting online-gambling (judi online) promotion

**Business question:** a platform wants to remove comments that promote online gambling sites ("judol"). Every comment is classified as **1 = judol promotion** or **0 = not judol**. This is binary classification with an adversary: promoters disguise brand names to avoid filters.

## 2.1 Peek, then load
The file is small (about 5 MB), so a single `read_csv` is fine, but we still peek first.

In [ ]:
display(pd.read_csv(JUDOL_PATH, nrows=3))
judol = pd.read_csv(JUDOL_PATH)
judol["message"] = judol["message"].astype(str)
judol["clean_message"] = judol["clean_message"].fillna("")
print(f"{len(judol):,} comments, columns: {list(judol.columns)}")
print("Missing:", judol.isna().sum().to_dict())

| Column | Meaning |
|---|---|
| `user` | display name of the commenter |
| `message` | original comment text |
| `label` | 1 = judol promotion, 0 = not |
| `source` | YouTube or Twitter |
| `clean_message` | a cleaned version provided with the dataset |

We will **not** use `user` as a feature: promoters create new accounts constantly, and a model that memorises account names would fail on tomorrow's accounts.

## 2.2 EDA

In [ ]:
print(judol["label"].value_counts().rename({0: "not judol (0)", 1: "judol (1)"}).to_string(), "\n")
display(pd.crosstab(judol["source"], judol["label"], margins=True))

The data is **exactly 50/50**, and also 50/50 inside each source. Real comment sections are nothing like this: judol comments are a small fraction. The dataset was clearly *sampled to be balanced*. That is fine for training, but precision measured here will be **optimistic** compared with production. We will quantify this in 2.8.

In [ ]:
FANCY_RE = re.compile(r"[\U0001D400-\U0001D7FF０-ｚⒶ-ⓩ]")      # math bold/italic, full-width, circled
EMOJI_RE = re.compile(r"[\U0001F300-\U0001FAFF☀-➿]")
SPACED_RE = re.compile(r"(?:\b\w ){3,}\w\b")                                       # "A L E X I S"

eda = pd.DataFrame({
    "label": judol["label"],
    "words": judol["message"].str.split().str.len(),
    "fancy font": judol["message"].str.contains(FANCY_RE),
    "spaced letters": judol["message"].apply(lambda s: bool(SPACED_RE.search(unicodedata.normalize("NFKC", s)))),
    "emoji": judol["message"].str.contains(EMOJI_RE),
    "has digits": judol["message"].str.contains(r"\d"),
    "has URL": judol["message"].str.contains("http"),
})
summary = eda.groupby("label")[["fancy font", "spaced letters", "emoji", "has digits", "has URL"]].mean().T
summary.columns = ["not judol", "judol"]
display((summary * 100).round(1).astype(str) + "%")
print(eda.groupby("label")["words"].describe()[["mean", "50%", "max"]].round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8))
x = np.arange(len(summary)); w = 0.38
ax.bar(x - w / 2, summary["not judol"], w, color=JUDOL_COLORS[0], label="not judol")
ax.bar(x + w / 2, summary["judol"], w, color=JUDOL_COLORS[1], label="judol")
ax.set_xticks(x, summary.index); ax.set_ylabel("share of comments"); ax.legend()
ax.set_title("Surface signals by class")
plt.tight_layout(); plt.show()

print("Examples of judol comments:")
for t in judol.loc[judol["label"] == 1, "message"].sample(8, random_state=3):
    print("  •", t.replace("\n", " ")[:120])

Judol comments almost always contain **digits** (brand names like *pluto88*, *alexis17*), and more than half use **"fancy" Unicode letters** (𝐏𝐋𝐔𝐓𝐎, 𝗃𝗉𝗍𝗈𝗀𝖾𝗅) or **spaced letters** (`A L E X I S 1 7`), almost never seen in normal comments. These tricks hide the brand name from simple keyword filters. They are also exactly what our preprocessing must handle.

## 2.3 Audit the provided `clean_message`
Never trust someone else's cleaning without checking it. Compare a few originals with the provided cleaned text:

In [ ]:
audit_ids = [i for i in [24034, 17259, 17253, 27512, 17024, 656] if i in judol.index]
display(judol.loc[audit_ids, ["message", "clean_message"]])

The provided cleaning **deleted the brand names**: `Keren euyy 𝗃𝗉𝗍𝗈𝗀𝖾𝗅77` became `keren euyy 77`, and `DI 🏮𝐏 𝐋 𝐔 𝐓 𝐎 𝟖 𝟖🏮` disappeared completely. It probably kept only ASCII characters, and fancy letters are not ASCII. The most informative words were removed before any model could see them.

## 2.4 Our preprocessing: undo the obfuscation
| Trick | Example | Fix |
|---|---|---|
| Fancy Unicode letters | `𝗃𝗉𝗍𝗈𝗀𝖾𝗅77` | Unicode **NFKC** normalisation → `jptogel77` |
| Combining marks | `P͟u͟l͟a͟u͟777` | remove underline/accent marks → `pulau777` |
| Spaced letters | `A L E X I S 1 7` | join runs of single characters → `alexis17` |
| Leetspeak inside words | `J3KP0T` | map digits to letters only *inside* mixed tokens → `jekpot` |
| Elongation | `sssaaaayyy` | reduce repeated letters |
| Handles | `@hauswin138` | keep the handle text (here the account *is* the advertisement), drop the `@` |
| URLs | `https://t.co/...` | replace with a `url` token (presence may matter) |

Unlike Case 1, we keep digits: `77`, `88`, `138` are part of brand names.

In [ ]:
LEET = str.maketrans({"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t", "@": "a", "$": "s"})
MIXED_RE = re.compile(r"\b[a-z]+[0-9@$][a-z0-9@$]*[a-z]\b")      # letters-digits-letters, e.g. j3kp0t

def normalize_judol(text):
    s = unicodedata.normalize("NFKC", str(text))
    s = "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")   # P͟u͟l͟a͟u͟ → Pulau
    s = re.sub(r"https?://\S+|www\.\S+", " url ", s)
    s = s.lower()
    s = SPACED_RE.sub(lambda m: m.group(0).replace(" ", ""), s)   # a l e x i s 1 7 → alexis17
    s = re.sub(r"(?<!\w)@(\w)", r"\1", s)                        # keep handle text, drop '@'
    s = MIXED_RE.sub(lambda m: m.group(0).translate(LEET), s)     # j3kp0t → jekpot
    s = re.sub(r"([a-z])\1{2,}", r"\1", s)                        # elongation
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

t0 = time.perf_counter()
judol["norm"] = judol["message"].map(normalize_judol)
print(f"Normalised {len(judol):,} comments in {time.perf_counter() - t0:.1f}s")
display(judol.loc[audit_ids, ["clean_message", "norm"]])

In [ ]:
# How often did the provided cleaning lose a brand-like token (letters followed by 2+ digits, e.g. pluto88)?
BRAND_RE = re.compile(r"\b[a-z]{3,}\d{2,}\b")
lost = [bool(set(BRAND_RE.findall(n)) - set(BRAND_RE.findall(c))) for n, c in zip(judol["norm"], judol["clean_message"])]
judol["brand_lost_in_clean_message"] = lost
print(judol.groupby("label")["brand_lost_in_clean_message"].mean().rename({0: "not judol", 1: "judol"}).map("{:.1%}".format))

### Label noise check
Labels for judol are usually assigned by keyword search plus a quick human pass. Let's look for "judol" comments with **no gambling cue at all** (no gambling vocabulary, no brand-like token, no fancy font).

In [ ]:
CUES = re.compile(r"\b(?:gacor|maxwin|max win|jackpot|jekpot|jp|slot|togel|toto|depo|deposit|wd|withdraw|cuan|scatter|"
                  r"spin|rtp|zeus|olympus|pragmatic|situs|daftar|bonus|hoki|menang|main di|link|petir|bandar)\b")
has_cue = judol["norm"].str.contains(CUES) | judol["norm"].str.contains(BRAND_RE) | eda["fancy font"]
suspicious = judol[(judol["label"] == 1) & ~has_cue]
print(f"Judol-labelled comments without any cue: {len(suspicious):,} ({len(suspicious) / (judol['label'] == 1).sum():.1%} of judol)")
display(suspicious.sample(8, random_state=SEED)[["message"]])

The list mixes two different things:
- **promotions our simple cue list missed**, for example brand names split by symbols (`ALEXIS⭐17⭐`, `ᆞMᆞᆞAᆞᆞNᆞᆞDᆞ…`), look-alike letters from other alphabets (`MΛNDALIKA77` uses a Greek Λ), or a brand hidden in a username. These are more obfuscation tricks, so the label is right and the *cue list* is wrong;
- **comments that look completely ordinary** (requests to the YouTuber, get-well wishes). These are probably **mislabelled**. In a real project you would send this list back to the annotators with a clear labeling guide (slides 6–7). Here we keep the labels as given and remember that part of our "errors" may be label errors.

## 2.5 Split
Exact duplicates after normalisation are removed first, then a stratified 80/20 split.

In [ ]:
before = len(judol)
judol_d = judol.drop_duplicates("norm").reset_index(drop=True)
judol_d = judol_d[judol_d["norm"].str.len() > 0]
print(f"Removed {before - len(judol_d)} duplicate/empty comments")

j_train, j_test = train_test_split(judol_d, test_size=0.2, stratify=judol_d["label"], random_state=SEED)
print(f"train {len(j_train):,} | test {len(j_test):,} | judol share train {j_train['label'].mean():.1%}, test {j_test['label'].mean():.1%}")
jy_train, jy_test = j_train["label"].values, j_test["label"].values

## 2.6 Baselines: a rule, then NB / LR / SVM
A keyword rule is the classic first attempt, so it is the baseline to beat. Then the three models from the slides, each on **two versions of the text** (the provided `clean_message` and our `norm`), with the same split and the same TF-IDF settings.

In [ ]:
def binary_scores(y_true, y_pred):
    return {"precision": precision_score(y_true, y_pred), "recall": recall_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred), "accuracy": accuracy_score(y_true, y_pred)}

rule_pred = (j_test["norm"].str.contains(CUES) | j_test["norm"].str.contains(BRAND_RE)
             | j_test["message"].str.contains(FANCY_RE)).astype(int).values
rows = [{"model": "Keyword / pattern rule", "text": "message", **binary_scores(jy_test, rule_pred)}]

word_tfidf = lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
models = {"Multinomial NB": lambda: MultinomialNB(), "Logistic Regression": lambda: LogisticRegression(max_iter=2000),
          "Linear SVM": lambda: LinearSVC()}
for text_col in ["clean_message", "norm"]:
    for name, make in models.items():
        p = Pipeline([("tfidf", word_tfidf()), ("clf", make())]).fit(j_train[text_col], jy_train)
        rows.append({"model": name, "text": text_col, **binary_scores(jy_test, p.predict(j_test[text_col]))})
word_results = pd.DataFrame(rows).set_index(["model", "text"])
display(word_results.style.format("{:.3f}"))

- The rule is fairly precise but misses about a quarter of judol comments: every new brand name or spelling needs a new rule.
- Every model improves when it gets **our normalised text** instead of the provided `clean_message`, because the brand names are back.
- Linear SVM is the strongest of the three on word features.

## 2.7 Character n-grams
Obfuscation creates endless spelling variants (`gacor`, `gac0r`, `gacorr`, `g4cor`). Word features treat each variant as a new, unseen word. **Character n-grams** (`analyzer="char_wb"`, substrings of length 2–5 within word boundaries) still share most pieces between variants, so they generalise better.

In [ ]:
char_tfidf = lambda: TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True,
                                     max_features=300_000)
rows = []
for text_col in ["clean_message", "norm"]:
    for name, make in [("Logistic Regression", lambda: LogisticRegression(max_iter=2000)), ("Linear SVM", lambda: LinearSVC())]:
        t0 = time.perf_counter()
        p = Pipeline([("tfidf", char_tfidf()), ("clf", make())]).fit(j_train[text_col], jy_train)
        rows.append({"model": name + " (char 2–5)", "text": text_col,
                     **binary_scores(jy_test, p.predict(j_test[text_col])), "seconds": time.perf_counter() - t0})
char_results = pd.DataFrame(rows).set_index(["model", "text"])
display(char_results.style.format("{:.3f}").format("{:.1f}", subset=["seconds"]))

Linear SVM with character n-grams on our normalised text gives the best precision **and** recall so far. Note that these test-set numbers are only for illustration. We pick the final configuration with cross-validation on the training set, below, and the test set is used once more at the end.

## 2.8 GridSearchCV and the metric that matches the cost of errors
Slide 27: *auto-hiding comments* means the costly error is **hiding an innocent comment** (a false positive), so precision matters most. A moderation queue reviewed by humans would instead prioritise recall.

We tune with **F0.5**, an F-score that weights precision more than recall (slide 26: β < 1 favours precision).

In [ ]:
f05 = make_scorer(fbeta_score, beta=0.5)
judol_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", min_df=3, sublinear_tf=True, max_features=300_000)),
    ("clf", LinearSVC()),
])
judol_grid = {"tfidf__ngram_range": [(2, 4), (2, 5)], "clf__C": [0.1, 1, 10]}
skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
judol_search = GridSearchCV(judol_pipe, judol_grid, scoring={"f05": f05, "precision": "precision", "recall": "recall"},
                            refit="f05", cv=skf5, n_jobs=-1)
t0 = time.perf_counter()
judol_search.fit(j_train["norm"], jy_train)
print(f"Grid search finished in {time.perf_counter() - t0:.0f}s | best: {judol_search.best_params_}")

res = pd.DataFrame(judol_search.cv_results_)
display(res[["param_tfidf__ngram_range", "param_clf__C", "mean_test_f05", "std_test_f05",
             "mean_test_precision", "mean_test_recall"]].sort_values("mean_test_f05", ascending=False)
        .style.format({c: "{:.3f}" for c in ["mean_test_f05", "std_test_f05", "mean_test_precision", "mean_test_recall"]}))

### Choosing the decision threshold
A linear SVM outputs a score; by default anything above 0 is "judol". Moving the threshold trades recall for precision (slide 39, question C). The threshold must be chosen on **training data** (with cross-validated predictions), never on the test set.

In [ ]:
best_judol = judol_search.best_estimator_
oof_scores = cross_val_predict(best_judol, j_train["norm"], jy_train, cv=skf5, method="decision_function", n_jobs=-1)
prec, rec, thr = precision_recall_curve(jy_train, oof_scores)

TARGET_PRECISION = 0.99
ok = np.where(prec[:-1] >= TARGET_PRECISION)[0]
threshold = thr[ok[0]] if len(ok) else 0.0
print(f"Threshold for ≥{TARGET_PRECISION:.0%} precision (cross-validated on train): {threshold:.3f} "
      f"→ expected recall {rec[ok[0]]:.1%}" if len(ok) else "Target precision not reachable")

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(rec, prec, color="#2a78d6", lw=2)
i0 = np.argmin(np.abs(thr - 0)); ax.scatter(rec[i0], prec[i0], color="#8a8984", zorder=3, label="default threshold (0)")
if len(ok): ax.scatter(rec[ok[0]], prec[ok[0]], color="#eb6834", zorder=3, label=f"threshold {threshold:.2f}")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0.8, 1.005); ax.legend(loc="lower left")
ax.set_title("Precision–recall trade-off (cross-validated, train)")
plt.tight_layout(); plt.show()

### Final evaluation on the test set (once)

In [ ]:
test_scores = best_judol.decision_function(j_test["norm"])
for name, th in [("default threshold 0", 0.0), (f"precision-first threshold {threshold:.2f}", threshold)]:
    p = (test_scores >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(jy_test, p).ravel()
    print(f"{name:32} TP={tp:5} FP={fp:4} FN={fn:4} TN={tn:5} | precision {tp / (tp + fp):.3f} recall {tp / (tp + fn):.3f} F1 {f1_score(jy_test, p):.3f}")

judol_pred = (test_scores >= threshold).astype(int)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ConfusionMatrixDisplay.from_predictions(jy_test, judol_pred, display_labels=["not judol", "judol"], ax=ax,
                                        cmap="Blues", colorbar=False, values_format=",")
ax.set_title("Test set, precision-first threshold")
plt.tight_layout(); plt.show()

### What happens at a realistic prevalence?
On a balanced test set, precision looks great. But if only 2% of real comments are judol, the same false-positive *rate* hits 49 times more innocent comments for every judol comment. Precision at prevalence π is

$$\text{precision} = \frac{\text{TPR}\cdot\pi}{\text{TPR}\cdot\pi + \text{FPR}\cdot(1-\pi)}$$

In [ ]:
rows = []
for name, th in [("default 0", 0.0), (f"precision-first {threshold:.2f}", threshold)]:
    p = (test_scores >= th).astype(int)
    tpr = recall_score(jy_test, p)
    fpr = ((p == 1) & (jy_test == 0)).sum() / (jy_test == 0).sum()
    for prevalence in [0.50, 0.10, 0.02]:
        rows.append({"threshold": name, "judol prevalence": f"{prevalence:.0%}",
                     "precision": tpr * prevalence / (tpr * prevalence + fpr * (1 - prevalence)),
                     "innocent comments hidden per 10k": fpr * (1 - prevalence) * 10_000})
display(pd.DataFrame(rows).pivot(index="judol prevalence", columns="threshold").reindex(["50%", "10%", "2%"]).round(3))

This is why the threshold is a **business decision** (slide 27): the platform must say how many wrongly hidden comments per 10,000 it accepts.

## 2.9 Error analysis and what the model learned

In [ ]:
errs = j_test.assign(score=test_scores, pred=judol_pred)
print("False positives (innocent comments that would be hidden):")
display(errs[(errs["pred"] == 1) & (errs["label"] == 0)].nlargest(6, "score")[["message", "score"]])
print("False negatives (judol comments that slip through):")
display(errs[(errs["pred"] == 0) & (errs["label"] == 1)].nsmallest(6, "score")[["message", "score"]])

Read the "errors" before blaming the model. Most of the top **false positives are obvious promotions labelled 0** (spaced or fancy-font brand names such as *PULAU777*, *ALEXIS 17*, *DORA77*): the model is right and the label is wrong. Many **false negatives contain no visible promotion** (a bare link, random strings, text in another script). Label noise in both directions caps the score we can measure.

In [ ]:
vec, svm = best_judol.named_steps["tfidf"], best_judol.named_steps["clf"]
feat = vec.get_feature_names_out()
order = np.argsort(svm.coef_[0])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, idx, color, title in [(axes[0], order[:15], JUDOL_COLORS[0], "→ not judol"),
                              (axes[1], order[-15:], JUDOL_COLORS[1], "→ judol")]:
    ax.barh([repr(f) for f in feat[idx]], np.abs(svm.coef_[0][idx]), color=color)
    ax.set_title(f"Strongest character n-grams {title}"); ax.set_xlabel("|weight|")
plt.tight_layout(); plt.show()

The strongest judol features are **numbers** (`77`, `88`), the endings of brand names. That works today, but it is fragile: a promoter can switch to a brand without digits, and an innocent comment like *"lomba 17 agustus"* gets pushed toward judol. Feature weights are a quick way to see what a model will be vulnerable to.

### Robustness test: new disguises
Promoters will try new tricks tomorrow. Compare a word-level model trained on the provided `clean_message` with our final character-level model on hand-written examples:

In [ ]:
word_model = Pipeline([("tfidf", word_tfidf()), ("clf", LinearSVC())]).fit(j_train["clean_message"], jy_train)
probes = ["gacor parah main di pluto88 auto wd",
          "g a c o r bgt hari ini, langsung maxwin",
          "𝓖𝓐𝓒𝓞𝓡 𝓜𝓐𝓧𝓦𝓘𝓝 di 𝓩𝓔𝓤𝓢𝟕𝟕",
          "cobain d1 ALEX1S17 dijamin j3kp0t",
          "videonya bermanfaat banget, makasih ilmunya bang",
          "alhamdulillah menang lomba 17 agustus di kampung",
          "link materinya ada di deskripsi ya"]
display(pd.DataFrame({"comment": probes,
                      "word model on clean_message": word_model.predict([p.lower() for p in probes]),
                      "final char model": (best_judol.decision_function([normalize_judol(p) for p in probes]) >= threshold).astype(int),
                      "char model score": best_judol.decision_function([normalize_judol(p) for p in probes]).round(2)}))

Neither model is perfect. The word model flags the harmless *"link materinya ada di deskripsi"*. The char model lets *"g a c o r bgt …"* through: after the letters are joined the comment has *gacor* and *maxwin* but no brand name or digits, and the model leans heavily on those (see the feature weights above). Adding more varied training examples, not just more rules, is the long-term fix.

✍️ Write your own disguised promotion and check whether it gets through. What preprocessing step would catch it?

In [ ]:
joblib.dump({"pipeline": best_judol, "threshold": float(threshold), "normalizer": "normalize_judol (see notebook)"},
            "judol_detector_char_svm.joblib")
print("Saved judol_detector_char_svm.joblib")

# Homework
Pick **one** and submit the notebook with a short written conclusion (max. half a page):

1. **Stemming and stopwords (Case 1).** Add Sastrawi stemming or stopword removal to `clean_review`. Re-run the grid search. Does macro-F1 change? Does Neutral recall change? Explain why.
2. **Aspect-based sentiment (Case 1).** Choose two aspects (for example *login/OTP* and *payment*). Write keyword rules to find reviews about each aspect, then report the sentiment distribution per aspect and per app. What would a ministry do with this?
3. **Prevalence-aware evaluation (Case 2).** Build a test set with 2% judol (sample from `j_test`). Report precision and recall at the default and the precision-first thresholds. Recommend a threshold to a platform manager in plain language.
4. **Relabel a sample.** Take 100 random reviews from Case 1, label them yourself using the labeling guide in 1.3, and compare with the star-based labels. How often do you disagree, and on which class?

---
### Checklist
- [ ] Peek at the data, then load only the columns you need, in compact types.
- [ ] Check where the labels come from and read examples of disagreement before modelling.
- [ ] Deduplicate **before** splitting; resolve or drop conflicting labels.
- [ ] Stratified train / validation / test; the test set is used once.
- [ ] Vectorizer and classifier in one `Pipeline`, so cross-validation never leaks.
- [ ] Compare baselines (dummy, NB, LR, SVM) on the same split and representation.
- [ ] Report per-class precision, recall, F1 and the confusion matrix; use macro-F1 for imbalanced multiclass.
- [ ] Choose the metric and threshold from the cost of errors, and check precision at realistic prevalence.
- [ ] Read the errors: they often reveal label noise or a preprocessing bug (like the deleted brand names).